In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import cftime
from xarray.coding.times import CFDatetimeCoder
import os


In [2]:
# -----------------------------
# CONFIG
# -----------------------------
base_path = Path("./data/")
year = 2003
month = 1
os.makedirs("./outputs/", exist_ok=True)
output_file = f"./outputs/POF_prediction_{year}_{month:02d}.nc"

# Load trained model
model = joblib.load("./data/POF_model.joblib")


In [3]:

# -----------------------------
# LOAD STATIC DATA
# -----------------------------
time_coder = CFDatetimeCoder(use_cftime=True)
PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc_2")
UR = xr.open_dataset(base_path / "CLIMATE/urban_C.nc_2", decode_times=time_coder)
RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_c.nc_2")

UR_arr = UR.vegdiff.squeeze().values  # keep 2D shape
PO_arr = PO.population_density.values  
RD_arr = RD.road_length.values         


In [6]:

# -----------------------------
# LOAD DYNAMIC DATA
# -----------------------------
mon = f"{month:02d}"
ds_paths = {
    "AF": f"ACTIVE_FIRE_MAP_{year}_{mon}_R.nc",
    "FU": f"FUEL_MAP_{year}_{mon}_R.nc",
    "DF": f"DFMC_MAP_{year}_{mon}_R.nc",
    "LF": f"LFMC_MAP_{year}_{mon}_R.nc",
    "PR": f"P_{year}_{mon}.nc",
    "T2": f"T2M_{year}_{mon}.nc",
    "D2": f"D2M_{year}_{mon}.nc",
    "WS": f"WS_{year}_{mon}.nc",
}
# Open all dynamic datasets
with xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
     xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
     xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
     xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
     xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
     xr.open_dataset(base_path / ds_paths["D2"]) as D2, \
     xr.open_dataset(base_path / ds_paths["WS"]) as WS, \
     xr.open_dataset(base_path / ds_paths["AF"]) as AF:

    n_days = len(AF.ACTIVE_FIRE)
    all_grids = []

    for i in range(n_days):
        # Extract arrays for timestep i
        FU_LL = FU.Live_Leaf[i].values
        FU_LW = FU.Live_Wood[i].values
        FU_DF = FU.Dead_Foliage[i].values
        FU_DW = FU.Dead_Wood[i].values
        DF_ = DF.DFMC_Foliage[i].values
        DW_ = DF.DFMC_Wood[i].values
        LF_ = LF.LFMC[i].values
        PR_ = PR.tp[i].values
        T2_ = T2.t2m[i].values
        D2_ = D2.d2m[i].values
        WS_ = WS.ws[i].values

        # Mask where total fuel > 0
        ft = FU_LL + FU_LW + FU_DF + FU_DW
        mask = ft > 0

        # Flatten and build dataframe for prediction
        feature_arrays = {
            "PR": PR_[mask],
            "T2": T2_[mask],
            "D2": D2_[mask],
            "WS": WS_[mask],
            "FU_LL": FU_LL[mask],
            "FU_LW": FU_LW[mask],
            "FU_DF": FU_DF[mask],
            "FU_DW": FU_DW[mask],
            "DF": DF_[mask],
            "DW": DW_[mask],
            "LF": LF_[mask],
            "UR": UR_arr[mask],
            "PO": PO_arr[mask],
            "RD": RD_arr[mask],
        }
        X_pred = pd.DataFrame(feature_arrays)

        # Predict probability
        y_proba = model.predict_proba(X_pred)[:, 1]

        # Create full grid and fill masked values
        fire_prob_grid = np.full(ft.shape, np.nan, dtype=float)
        fire_prob_grid[mask] = y_proba
        all_grids.append(fire_prob_grid)

    # Stack all timesteps into 3D array (time, lat, lon)
    fire_prob_array = np.stack(all_grids, axis=0)

    # -----------------------------
    # SAVE TO NETCDF
    # -----------------------------
    time = [cftime.DatetimeJulian(year, month, day+1) for day in range(n_days)]
    ds_out = xr.Dataset(
        {"fire_probability": (["time", "lat", "lon"], fire_prob_array)},
        coords={
            "time": time,
            "latitude": PO.latitude,
            "longitude": PO.longitude
        }
    )

    ds_out.to_netcdf(output_file)
    print(f"✅ Prediction saved → {output_file}")

IndexError: boolean index did not match indexed array along axis 1; size of axis is 3600 but size of corresponding boolean axis is 5136